In [35]:
import pandas as pd
import numpy as np
import cobra
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import requests
from cobra import Model, Reaction, Metabolite


In [79]:
rxns = pd.read_csv("../../results/pbi_model_gapfill/reactions_added_metabolomics.csv")["RXN"]
rxns = rxns.dropna().str.strip()
rxns

# read in the model
model = cobra.io.json.load_json_model("../../results/pbi_model_gapfill/bifermentans_gapfilled_from_cdiff.json")

# read in modelSEED data
modelseed_reactions = pd.read_csv("../../data/model_seed_database/ModelSEEDDatabase/Biochemistry/reactions.tsv", sep="\t", index_col = 0)

rxns = [i for i in rxns if i in modelseed_reactions.index.values]


def parse_stoichiometry(stoich_string):
    stoich_dict = {}
    print(stoich_string)
    for compound in stoich_string.strip("'").split(";"):
        parts = compound.split(":")
        if len(parts) >= 2:
            stoich_dict[parts[1]] = float(parts[0])

    return(stoich_dict)

stoichs = [parse_stoichiometry(i) for i in modelseed_reactions.loc[rxns, 'stoichiometry']]
print("# of reactions in model: ", len(model.reactions))




-1:cpd00022:0:0:"Acetyl-CoA";-1:cpd00023:0:0:"L-Glutamate";1:cpd00010:0:0:"CoA";1:cpd00067:0:0:"H+";1:cpd00477:0:0:"N-Acetyl-L-glutamate"
-1:cpd00017:0:0:"S-Adenosyl-L-methionine";-1:cpd11461:0:0:"DNA";1:cpd00019:0:0:"S-Adenosyl-homocysteine";1:cpd12223:0:0:"DNA 5-methylcytosine"
-1:cpd00003:0:0:"NAD";-1:cpd02120:0:0:"Dihydrodipicolinate";1:cpd00004:0:0:"NADH";1:cpd00067:0:0:"H+";1:cpd15596:0:0:"Dipicolinate"
-1:cpd00004:0:0:"NADH";-2:cpd00067:0:0:"H+";-1:cpd02091:0:0:"Imidazole pyruvate";1:cpd00003:0:0:"NAD";1:cpd03300:0:0:"Imidazole lactate"
-1:cpd00024:0:0:"2-Oxoglutarate";-1:cpd00119:0:0:"L-Histidine";1:cpd00023:0:0:"L-Glutamate";1:cpd02091:0:0:"Imidazole pyruvate"
-1:cpd00001:0:0:"H2O";-1:cpd00003:0:0:"NAD";-1:cpd00486:0:0:"Indoleacetaldehyde";1:cpd00004:0:0:"NADH";2:cpd00067:0:0:"H+";1:cpd00703:0:0:"Indoleacetate"
-1:cpd00067:0:0:"H+";-1:cpd00278:0:0:"Indolepyruvate";1:cpd00011:0:0:"CO2";1:cpd00486:0:0:"Indoleacetaldehyde"
-1:cpd00024:0:0:"2-Oxoglutarate";-1:cpd00065:0:0:"L-Trypt

In [80]:
# add reactions to the model
already_present = []
for rxn, stoich in zip(rxns, stoichs):
    if rxn not in model.reactions:
        new_reaction = Reaction(rxn)
        new_reaction.name = modelseed_reactions.loc[rxn, 'name']
        if pd.isna(new_reaction.name):
            new_reaction.name = "placeholder"
        new_reaction.lower_bound = 0.0  # Assuming default lower bound
        new_reaction.upper_bound = 1000.0  # Assuming default upper bound
        new_reaction.add_metabolites({Metabolite(met_id): coeff for met_id, coeff in stoich.items()})
        model.add_reactions([new_reaction])
    else:
        already_present.append(rxn)
print("# of reactions in model: ", len(model.reactions))

print(already_present)
cobra.io.json.save_json_model(model, "../../results/pbi_model_gapfill/bifermentans_gapfilled_from_cdiff_metabolomics.json")


# of reactions in model:  1730
['rxn00192']
